# Step-by-step demo of NEDAS using the vort2d model

## Load NEDAS and dependencies

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from NEDAS import get_scheme
from NEDAS.config import Config

## Initialize configuration and main scheme

In [19]:
config = Config(config_file="vort2d/config.yml")

config.debug = False
config.call_stack_max_level = None
config.nproc = 1
config.io_mode = 'offline'

In [20]:
scheme = get_scheme(config)

model = scheme.c.models['vort2d']
dataset = scheme.c.datasets['vort2d']


original_generate_ic = model.generate_initial_condition
def patched_generate_ic(*args, **kwargs):
    np.random.seed()
    return original_generate_ic(*args, **kwargs)
model.generate_initial_condition = patched_generate_ic

In [21]:
model.loc_sprd = 30000

In [34]:
grid = scheme.c.grid

In [35]:
from NEDAS.utils.spatial_operation import gradx, grady
def to_vorticity(vel):
    u, v = vel[0], vel[1]
    zeta = gradx(v, grid.dx, grid.cyclic_dim) - grady(u, grid.dy, grid.cyclic_dim)
    return zeta


## Prepare truth

check nc files in vort2d/truth

In [23]:
%rm -rf vort2d/truth/*

In [24]:
scheme.prepare_truth()


Generate vort2d truth ─────────────────────────── ⏳  [──────────]   6%                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         566086.4393634556 617825.2672544577
Generate vort2d truth ─────────────────────────── ✅    22.75s                                                                                                       

In [25]:
%rm -rf vort2d/init_run/*

In [ ]:
np.random.seed()
scheme.prepare_init_ensemble()

In [27]:
scheme.c.time = scheme.c.config.time_start

truth_state = []
while scheme.c.time < scheme.c.config.time_end:
    var = model.read_var(path = model.truth_dir, name='velocity', member=None, time=scheme.c.time)
    truth_state.append(var)
    scheme.c.time = scheme.c.next_time

In [28]:
time = scheme.c.config.time_start

init_ens_state = []
for m in range(scheme.c.nens):
    init_ens_state.append(model.read_var(path=model.ens_init_dir, name='velocity', member=m, time=time))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5))
grid = scheme.c.grid
n = 0
for m in range(scheme.c.nens):
    ax.contour(grid.x, grid.y, to_vorticity(init_ens_state[m]), [1e-3], colors='c')

ax.contour(grid.x, grid.y, to_vorticity(truth_state[n]), [1e-3], colors='k')

#grid.plot_vectors(ax, truth_state[n])